In [1]:
import sys
import warnings
import numpy as np
import pathlib as pl
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern
from scipy.stats import norm
from scipy.optimize import minimize
root_folder = pl.Path.cwd().parents[2]
sys.path.insert(0, str(root_folder / "utilities"))
import common_functions as cf

warnings.filterwarnings('ignore')

initial_data_folder = "data/initial_data/function_7"
initial_inputs_path = pl.Path.joinpath(root_folder, initial_data_folder,  "initial_inputs.npy")
initial_outputs_path = pl.Path.joinpath(root_folder, initial_data_folder, "initial_outputs.npy")

In [2]:
data_in = np.load(initial_inputs_path)
data_out = np.load(initial_outputs_path)

Week-01

In [3]:
X_init = data_in
y_init = data_out

y_best = y_init.max()  # best performance so far

kernel = Matern(nu=2.5)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, normalize_y=True)
gp.fit(X_init, y_init)

# --- Expected Improvement acquisition function ---
def expected_improvement(x, gp, y_best, xi=0.01):
    x = np.array(x).reshape(1, -1)
    mu, sigma = gp.predict(x, return_std=True)
    mu, sigma = mu[0], sigma[0]
    if sigma == 0.0:
        return 0.0
    imp = mu - y_best - xi
    Z = imp / sigma
    ei = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
    return -ei  # negative for minimization in scipy

# --- Optimize acquisition function ---
bounds = [(0,1)]*6
best_x = None
best_ei = float('inf')

for _ in range(20):  # multiple random starts
    x0 = np.random.rand(6)
    res = minimize(lambda x: expected_improvement(x, gp, y_best),
                   x0=x0, bounds=bounds, method='L-BFGS-B')
    if res.fun < best_ei:
        best_ei = res.fun
        best_x = res.x

x_next = best_x
print(cf.format_inputdata(x_next))

0.197786-0.736062-0.302031-0.770104-0.259471-0.221691


Week-02

In [4]:
X_new = np.array([
    [0.000000, 0.210370, 1.000000, 0.000000, 0.323749, 1.000000]
])

y_new = np.array([ 
    0.543430844426678
])

# Combine all data
X_init = np.vstack((data_in, X_new))
y_init = np.concatenate((data_out, y_new))

y_best = y_init.max()  # best performance so far

In [5]:
# --- Fit Gaussian Process ---
kernel = Matern(nu=2.5)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, normalize_y=True)
gp.fit(X_init, y_init)
y_best = y_init.max()  # still the best observed performance

In [6]:
def expected_improvement(x, gp, y_best, xi=0.05):
    x = np.array(x).reshape(1, -1)
    mu, sigma = gp.predict(x, return_std=True)
    mu, sigma = mu[0], sigma[0]
    if sigma == 0:
        return 0
    imp = mu - y_best - xi
    Z = imp / sigma
    ei = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
    return -ei  # negative for minimize


In [7]:
bounds = [(0, 1)] * 6
best_x = None
best_ei = float('inf')

for _ in range(20):
    x0 = np.random.rand(6)
    res = minimize(lambda x: expected_improvement(x, gp, y_best),
                   x0=x0, bounds=bounds, method='L-BFGS-B')
    if res.fun < best_ei:
        best_ei = res.fun
        best_x = res.x

x_next = best_x
print(cf.format_inputdata(x_next))

0.175510-0.559570-0.701429-0.914288-0.312111-0.556274
